# HybridAgent — Exploración interactiva

Este notebook carga el paclet HVA y recorre las funciones principales de `HybridAgent`:
`AgentId`, accessors, actualizaciones inmutables y hash estructural.

## 1 · Cargar el paclet

In [1]:
PacletDirectoryLoad["/workspaces/hva-framework/paclet"];
Needs["HVA`"]

## 2 · Construir un agente de prueba

In [2]:
(* Agente termostat mínimo *)
Spec = HybridAgent["thermostat",
  Modes            -> {"off", "on"},
  ContinuousVars   -> {temp},
  VectorFields     -> <|
    "off" -> {temp'[t] == 0},
    "on"  -> {temp'[t] == 5 - temp[t]}
  |>,
  Transitions      -> {
    <|"from" -> "off", "to" -> "on",  "condition" -> temp < 18|>,
    <|"from" -> "on",  "to" -> "off", "condition" -> temp > 22|>
  },
  ModeInvariants   -> {0 <= temp <= 30},
  InitialMode      -> "off",
  TimeSymbol       -> t,
  InitialValuation -> <|temp -> 15|>
]

HybridAgent["thermostat" @ off]

In [3]:
(* Verificar que se construyó correctamente *)
HybridAgentQ[Spec]

True

## 3 · `AgentId` — identificador del agente

```
AgentId::usage = "AgentId[a] devuelve el identificador del agente."
```

In [4]:
AgentId[Spec]

thermostat

In [5]:
#(* Verificar tipo de retorno *)
StringQ[AgentId[Spec]]

True

In [6]:
#(* Error handling: non-HybridAgent *)
AgentId["not an agent"]

Expected HybridAgent.

## 4 · Accessors completos

In [7]:
AgentModes[Spec]

{off, on}

In [8]:
AgentContinuousVars[Spec]

{temp}

In [9]:
AgentVectorFields[Spec]

<|off -> {temp'[t] == 0}, on -> {temp'[t] == 5 - temp[t]}|>

In [10]:
AgentTransitions[Spec]

>    <|from -> on, to -> off, condition -> temp > 22, action -> Null|>}


{<|from -> off, to -> on, condition -> temp < 18, action -> Null|>,

In [11]:
AgentCurrentMode[Spec]

off

In [12]:
AgentValuation[Spec]

<|temp -> 15|>

In [13]:
AgentValuation[Spec][temp]

15

## 5 · Actualizaciones inmutables

Cada `With*` devuelve un **nuevo** agente sin modificar el original.

In [14]:
agent2 = WithCurrentMode[Spec, "on"];
{AgentCurrentMode[Spec], AgentCurrentMode[agent2]}

{off, on}

In [15]:
Spec3 = WithValuation[agent2, <|temp -> 20|>];
AgentValuation[Spec3][temp]

20

In [16]:
Spec4 = AppendTrace[Spec3, "transitioned to on"];
AgentTrace[Spec4]

{transitioned to on}

## 6 · Hash estructural

`AgentStructuralHash` es determinístico e ignora campos runtime (`currentState`, `valuation`, `mailbox`, `trace`).

In [17]:
#(* Mismo hash en dos instancias con distintos runtime fields *)
h1 = AgentStructuralHash[Spec3];
h2 = AgentStructuralHash[Spec4];
{h1, h2, h1 === h2}

>    174907470186684077924995982141523805616, True}


{174907470186684077924995982141523805616,

In [18]:
#(* Hash cambia si se modifica la estructura *)
SpecMod = HybridAgent["thermostat",
  Modes            -> {"off", "on", "standby"},
  ContinuousVars   -> {temp},
  VectorFields     -> <|
    "off"     -> {temp'[t] == 0},
    "on"      -> {temp'[t] == 5 - temp[t]},
    "standby" -> {temp'[t] == -0.1 * temp[t]}
  |>,
  Transitions      -> {},
  ModeInvariants   -> {},
  InitialMode      -> "off",
  TimeSymbol       -> t,
  InitialValuation -> <|temp -> 15|>
];
AgentStructuralHash[Spec3] =!= AgentStructuralHash[SpecMod]

True

## 7 · Información completa del agente

`Dataset[agent[[1]]]` muestra todos los campos canónicos como tabla interactiva.

In [19]:
(* agentOn: termostat con modo actual = "on" *)
agentOn = WithCurrentMode[Spec, "on"];
Dataset[agentOn[[1]]]

>     continuousVars -> {temp}, time -> t, 
>     vectorFields -> 
>      <|off -> {temp'[t] == 0}, on -> {temp'[t] == 5 - temp[t]}|>, 
>     transitions -> 
>      {<|from -> off, to -> on, condition -> temp < 18, action -> Null|>, 
>       <|from -> on, to -> off, condition -> temp > 22, action -> Null|>}, 
>     modeInvariants -> {0 <= temp, temp <= 30}, 
>     contract -> Contract[<|assumes -> {}, guarantees -> {}|>], 
>     rewriteRules -> {}, initialMode -> off, 
>     initialValuation -> <|temp -> 15|>, currentMode -> on, 
>     valuation -> <|temp -> 15|>, mailbox -> {}, trace -> {}|>, 
>    TypeSystem`Struct[{id, modes, continuousVars, time, vectorFields, 
>      transitions, modeInvariants, contract, rewriteRules, initialMode, 
>      initialValuation, currentMode, valuation, mailbox, trace}, 
>     {TypeSystem`Atom[String], 
>      TypeSystem`Vector[TypeSystem`Atom[String], 2], 
>      TypeSystem`Vector[TypeSystem`AnyType, 1], TypeSystem`AnyType, 
>      TypeSystem`Assoc[Typ

Dataset[<|id -> thermostat, modes -> {off, on},

In [20]:
(* Vista de transiciones como Dataset *)
Dataset[AgentTransitions[agentOn]]

>      action -> Null|>, <|from -> on, to -> off, condition -> temp > 22, 
>      action -> Null|>}, TypeSystem`Vector[TypeSystem`Struct[{from, to, 
>       condition, action}, {TypeSystem`Atom[String], 
>       TypeSystem`Atom[String], TypeSystem`AnyType, TypeSystem`AnyType}], 2]\
>     , <||>]


Dataset[{<|from -> off, to -> on, condition -> temp < 18,